# Single-Protein DDPM Training

This notebook trains a DDPM model on **one specific protein** by learning to denoise different conformational states (frames) from an MD trajectory.

**Workflow:**
1. Specify protein name and parameters
2. Dynamically create/load trajectory data for that protein
3. Train DDPM to denoise frames
4. Generate and evaluate results

## Step 1: Environment Setup

In [1]:
# Check environment
try:
    import google.colab
    IN_COLAB = True
    print("Running on Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

# Install dependencies (Colab only — locally assume deps are already installed)
if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio'], check=True)
    subprocess.run(['pip', 'install', '-q', 'omegaconf', 'pandas', 'tqdm', 'numpy', 'matplotlib', 'lightning', 'mdtraj', 'requests'], check=True)

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Running locally
PyTorch 2.5.1
CUDA available: False


In [2]:
import os

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Persist data on Drive so it survives session restarts
    os.makedirs('/content/drive/MyDrive/protein_data/data', exist_ok=True)

    if not os.path.exists('data'):
        os.symlink('/content/drive/MyDrive/protein_data/data', 'data')
    print("Drive mounted; data/ -> /content/drive/MyDrive/protein_data/data")
else:
    # Running locally: just make sure data/ exists
    os.makedirs('data', exist_ok=True)
    print(f"Local mode; data dir: {os.path.abspath('data')}")

Local mode; data dir: /home/jiwonjjeong/repos/winter-gen-pproject/data


## Step 2: Get Code from GitHub

Clone your repository to get the `gen_model` code (no data needed)

In [3]:
import os, subprocess

REPO_URL = "https://github.com/JiwonJJeong/winter-gen-pproject.git"

if IN_COLAB:
    if not os.path.exists('winter-gen-pproject'):
        print(f"Cloning {REPO_URL} ...")
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('winter-gen-pproject')
    # Pull latest changes and clear bytecode cache
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
    for cmd in [
        'find gen_model -name "*.pyc" -delete',
        'find gen_model -name "__pycache__" -type d -exec rm -rf {} + 2>/dev/null; true',
    ]:
        subprocess.run(cmd, shell=True)
    print("Code updated")
else:
    # Already inside the repo; just verify gen_model/ is present
    if not os.path.isdir('gen_model'):
        raise RuntimeError(
            "gen_model/ not found. Run this notebook from the repository root:\n"
            "  cd /path/to/winter-gen-pproject && jupyter notebook"
        )
    print(f"Local repo root: {os.path.abspath('.')}")

import subprocess as _sp
result = _sp.run(['ls', 'gen_model/'], capture_output=True, text=True)
print(result.stdout)
print("Code ready")

Local repo root: /home/jiwonjjeong/repos/winter-gen-pproject
README.md
__init__.py
__pycache__
all_atom.py
dataset.py
diffusion
ema.py
geometry.py
inference_se3.py.bak
models
parsing.py
residue_constants.py
rigid_utils.py
simple_inference.py
simple_train.py
split.py
splits
tensor_utils.py
test_dataset.py
train_se3.py.bak
utils.py

Code ready


## Step 3: Configure Protein and Training

**Customize these settings for your protein:**

In [ ]:
from omegaconf import OmegaConf

# ── HPO / LoRA settings ───────────────────────────────────────────────────────
# These are used by Step 8 (HPO) and Step 7 (model creation).
HPO_CONFIG = dict(
    lora_r             = 8,      # LoRA rank: 0 = full fine-tuning, 4/8/16 = LoRA adapters
    lora_alpha         = 0.0,    # 0 → auto-set to 2×lora_r
    n_trials           = 20,     # Optuna trials to run during HPO
    epochs_per_trial   = 5,      # Short epochs per trial (keep small for speed)
    pruner             = 'asha', # 'none' or 'asha' (Asynchronous Successive Halving)
)

# ── Protein / training settings ───────────────────────────────────────────────
protein_config = OmegaConf.create({
    'protein': {
        'name':              '4o66_C',
        'replica':           1,
        'train_early_ratio': 0.3,
        'train_ratio':       0.4,
        'val_ratio':         0.2,
    },
    'training': {
        'batch_size':        8,
        'num_epochs':        100,
        'learning_rate':     1e-4,
        'rot_loss_weight':   1.0,
        'trans_loss_weight': 1.0,
        'psi_loss_weight':   1.0,
    },
    'inference': {
        'num_samples': 5,
        'num_steps':   200,
    },
})
prot_cfg = protein_config.protein

# SE(3) diffusion and model configs are built from default_se3_conf() /
# default_model_conf() in later cells. No coordinate_scaling needed here:
# MDGenDataset computes a dynamic coord_scale from training data statistics.

print("Configuration Summary:")
print("="*70)
print(f"Protein   : {prot_cfg.name}_R{prot_cfg.replica}")
print(f"LoRA      : r={HPO_CONFIG['lora_r']}  (0 = full fine-tuning)")
print(f"HPO       : {HPO_CONFIG['n_trials']} trials × {HPO_CONFIG['epochs_per_trial']} epochs | pruner={HPO_CONFIG['pruner']}")
print(f"Training  : {protein_config.training.num_epochs} epochs | batch={protein_config.training.batch_size} | lr={protein_config.training.learning_rate}")
print("="*70)

## Step 4: Create/Load Protein Data

This creates the data structure for your specified protein

In [5]:
import os

# Use the automatic download and prep script
prot_name = protein_config.protein.name
!python scripts/download_and_prep.py {prot_name} --data_dir ./data --out_dir ./data --suffix _latent

# Setup paths for verification
prot_cfg = protein_config.protein
PROTEIN_FULL_NAME = f"{prot_cfg.name}_R{prot_cfg.replica}"
protein_dir = f'data/{prot_cfg.name}'
traj_path = f'{protein_dir}/{PROTEIN_FULL_NAME}_latent.npy'

if os.path.exists(traj_path):
    print(f"✅ Data ready at: {traj_path}")
else:
    print(f"❌ Error: Data not found at {traj_path}")


Data for 4o66_C seems to exist at ./data/4o66_C. Skipping download.
Detected timestep: 10.0 ps
Preprocessing output ./data/4o66_C/4o66_C_R1_latent.npy already exists. Skipping.
Preprocessing output ./data/4o66_C/4o66_C_R2_latent.npy already exists. Skipping.
Preprocessing output ./data/4o66_C/4o66_C_R3_latent.npy already exists. Skipping.
Creating 4-way split (Early: 5.0ns, Ratios: [0.6, 0.2, 0.2])...
Found 3 trajectory file(s) for 4o66_C.
  4o66_C_R1_latent: Total 10001, Timestep 10.0ps
    Early: [0:500], Train: [500:6200], Val: [6200:8100], Test: [8100:10001]
  4o66_C_R2_latent: Total 10001, Timestep 10.0ps
    Early: [0:500], Train: [500:6200], Val: [6200:8100], Test: [8100:10001]
  4o66_C_R3_latent: Total 10001, Timestep 10.0ps
    Early: [0:500], Train: [500:6200], Val: [6200:8100], Test: [8100:10001]
Updated splits saved to gen_model/splits/frame_splits.csv
✅ Data ready at: data/4o66_C/4o66_C_R1_latent.npy


In [ ]:

# =============================================================================
# Reference Trajectory Visualization
# Load a sample of MD frames and show them inline before training.
# =============================================================================
import numpy as np, pandas as pd, os
from IPython.display import display
from gen_model.utils.structure_utils import save_ca_trajectory_pdb, as_mdtraj_trajectory

# Load residue sequence — also used by the inline inference cell below
_seq_df = pd.read_csv('gen_model/splits/atlas.csv', index_col='name')
seqres  = _seq_df.loc[prot_cfg.name, 'seqres']
print(f"Protein : {prot_cfg.name}  ({len(seqres)} residues)")

# Sample 30 evenly-spaced Cα frames from the raw MD trajectory
N_REF  = 30
_arr   = np.lib.format.open_memmap(traj_path, 'r')
_idxs  = np.linspace(0, len(_arr) - 1, N_REF, dtype=int)
ref_ca = _arr[_idxs, :, 1, :].astype(np.float32)        # [N_REF, N, 3]
ref_ca = ref_ca - ref_ca.mean(axis=(0, 1), keepdims=True)  # centre at origin

os.makedirs('outputs', exist_ok=True)
ref_pdb = f'outputs/{prot_cfg.name}_reference.pdb'
save_ca_trajectory_pdb(ref_ca, ref_pdb, seqres)
print(f"Reference PDB saved : {ref_pdb}  ({N_REF} frames)")
print("(Open in PyMOL / VMD / UCSF ChimeraX, or view inline below)")

try:
    import nglview as nv
    _traj_ref = as_mdtraj_trajectory(ref_ca, seqres)
    view_ref  = nv.show_mdtraj(_traj_ref)
    view_ref.add_representation('cartoon', selection='protein', color='blue')
    view_ref.center()
    print("\n▶ Reference MD frames (blue) — use the player bar to animate:")
    display(view_ref)
except ImportError:
    print("\nnglview not installed — install with:  pip install nglview")
    print(f"Then re-run this cell for inline 3-D viewing.")


## Step 5: Configure Dataset and Model

In [ ]:
import sys
sys.path.insert(0, '.')

from gen_model.train_unconditional    import SE3Module
from gen_model.train_base             import default_se3_conf, default_model_conf
from gen_model.data.dataset           import MDGenDataset
from gen_model.diffusion.se3_diffuser import SE3Diffuser

print("✓ Modules imported")

data_config = OmegaConf.create({
    'data_dir':       'data',
    'atlas_csv':      'gen_model/splits/atlas.csv',
    'train_split':    'gen_model/splits/frame_splits.csv',
    'suffix':         '_latent',
    'frame_interval': None,
    'crop_ratio':     0.95,
    'min_t':          0.01,
    'pep_name':       prot_cfg.name,    # single-protein filter
    'replica':        prot_cfg.replica, # single-replica filter
})

print(f"\nDataset config:")
print(f"  Protein : {data_config.pep_name}_R{data_config.replica}")

## Step 6: Load Dataset

In [ ]:
# Build SE3 diffuser from canonical defaults (schedule_gamma=1.0, no coordinate_scaling)
print("Initialising SE3Diffuser...")
se3_conf     = default_se3_conf()
se3_diffuser = SE3Diffuser(se3_conf)
print("✓ SE3Diffuser ready\n")

# MDGenDataset computes coord_scale dynamically from training data (~1/std of Ca coords)
print("Loading datasets...")
train_dataset = MDGenDataset(
    args=data_config, diffuser=se3_diffuser, mode='train',
    repeat=1, num_consecutive=1, stride=1,
)
val_dataset = MDGenDataset(
    args=data_config, diffuser=se3_diffuser, mode='val',
    repeat=1, num_consecutive=1, stride=1,
)
val_dataset.coord_scale = float(train_dataset.coord_scale)

print(f"\n✓ Training frames  : {len(train_dataset)}")
print(f"✓ Validation frames: {len(val_dataset)}")
print(f"✓ coord_scale      : {train_dataset.coord_scale:.4f}  (computed from data)")

sample     = train_dataset[0]
n_residues = sample['res_mask'].shape[0]
print(f"\nSample keys   : {list(sample.keys())}")
print(f"Residues      : {n_residues}")
print(f"rigids_t shape: {sample['rigids_t'].shape}  (per-residue SE3 frame at time t)")
print(f"t (noise)     : {sample['t']:.4f}")

## Step 7: Create Model

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}\n")

# Build model config with optional LoRA adapters
model_conf = default_model_conf(
    lora_r     = HPO_CONFIG['lora_r'],
    lora_alpha = HPO_CONFIG['lora_alpha'],
)

model_pl = SE3Module(
    model_conf        = model_conf,
    se3_conf          = se3_conf,
    lr                = protein_config.training.learning_rate,
    rot_loss_weight   = protein_config.training.rot_loss_weight,
    trans_loss_weight = protein_config.training.trans_loss_weight,
    psi_loss_weight   = protein_config.training.psi_loss_weight,
)

total     = sum(p.numel() for p in model_pl.parameters())
trainable = sum(p.numel() for p in model_pl.parameters() if p.requires_grad)
print(f"✓ Score Network: {total:,} total params | {trainable:,} trainable ({100*trainable/total:.1f}%)")
print(f"  LoRA r={HPO_CONFIG['lora_r']} → {'LoRA adapters only' if HPO_CONFIG['lora_r'] > 0 else 'full fine-tuning'}")

## Step 8: Train

In [9]:
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from torch.utils.data import DataLoader

# 1. DataLoaders
train_loader = DataLoader(train_dataset, batch_size=protein_config.training.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=protein_config.training.batch_size, shuffle=False)

# 2. Checkpointing
checkpoint_callback = ModelCheckpoint(
    dirpath=f'checkpoints/{prot_cfg.name}_se3',
    filename='se3-{epoch:02d}-{val_loss:.4f}',
    save_top_k=3,
    monitor='val_loss',
    mode='min',
    save_last=True,
)

# 3. Train  (model_pl created in Step 7)
trainer = L.Trainer(
    max_epochs=protein_config.training.num_epochs,
    accelerator="auto",
    devices=1,
    callbacks=[checkpoint_callback],
    precision="16-mixed" if torch.cuda.is_available() else 32,
)

trainer.fit(model_pl, train_dataloaders=train_loader, val_dataloaders=val_loader)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
/home/jiwonjjeong/miniforge3/envs/winter-gen-pproject/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.



  | Name  | Type         | Params | Mode  | FLOPs
-------------------------------------------------------
0 | model | ScoreNetwork | 9.9 M  | train | 0    
-------------------------------------------------------
9.9 M     Trainable params
0         Non-trainable params
9.9 M     Total params
39.425    Total estimated model params size (MB)
227       Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jiwonjjeong/miniforge3/envs/winter-gen-pproject/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

...

/home/jiwonjjeong/miniforge3/envs/winter-gen-pproject/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 0:   0%|          | 3/713 [00:18<1:14:26,  0.16it/s, v_num=7, train_loss_step=nan.0]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/jiwonjjeong/miniforge3/envs/winter-gen-pproject/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
RUN_HPO_INLINE = False  # set True to run HPO in this cell

if RUN_HPO_INLINE:
    import types, os, optuna
    from gen_model.hpo import make_objective, _build_pruner

    hpo_args = types.SimpleNamespace(
        mode                  = 'unconditional',
        data_dir              = 'data',
        atlas_csv             = 'gen_model/splits/atlas.csv',
        train_split           = 'gen_model/splits/frame_splits.csv',
        suffix                = '_latent',
        batch_size            = protein_config.training.batch_size,
        epochs_per_trial      = HPO_CONFIG['epochs_per_trial'],
        save_dir              = f'checkpoints/hpo/{prot_cfg.name}',
        pruner                = HPO_CONFIG['pruner'],
        asha_min_resource     = 2,
        asha_reduction_factor = 3,
    )
    os.makedirs(hpo_args.save_dir, exist_ok=True)

    pruner = _build_pruner(hpo_args)
    study  = optuna.create_study(
        direction      = 'minimize',
        storage        = f'sqlite:///{hpo_args.save_dir}/optuna.db',
        study_name     = f'se3_{prot_cfg.name}',
        load_if_exists = True,
        pruner         = pruner,
    )
    study.optimize(make_objective(hpo_args), n_trials=HPO_CONFIG['n_trials'])

    best = study.best_trial
    print(f"\nBest val_loss : {best.value:.6f}  (trial #{best.number})")
    print("Best params:")
    for k, v in best.params.items():
        print(f"  {k:25s}: {v}")
else:
    print("HPO skipped (RUN_HPO_INLINE=False).")
    print(f"To run from terminal:")
    print(f"  python gen_model/hpo.py --data_dir data"
          f" --n_trials {HPO_CONFIG['n_trials']}"
          f" --epochs_per_trial {HPO_CONFIG['epochs_per_trial']}"
          f" --pruner {HPO_CONFIG['pruner']}"
          f" --save_dir checkpoints/hpo")

import glob

ckpt_dir  = f'checkpoints/{prot_cfg.name}_se3'
ckpt_list = sorted(glob.glob(f'{ckpt_dir}/*.ckpt'))

if ckpt_list:
    print(f"Checkpoints in {ckpt_dir}/:")
    for p in ckpt_list:
        print(f"  {p}")
    # Prefer non-last checkpoints (ranked by val_loss in filename)
    ranked = [p for p in ckpt_list if 'last' not in p]
    best_ckpt = ranked[-1] if ranked else ckpt_list[-1]
    print(f"\nBest checkpoint : {best_ckpt}")
else:
    best_ckpt = None
    print(f"No checkpoints found in {ckpt_dir}/")
    print("Run Step 8 (training) first.")

## Step 9: Evaluate

In [ ]:

# =============================================================================
# Inline Inference + Visualization
# Runs the reverse SE(3) SDE inside the notebook and shows generated
# conformations alongside the reference MD frames.
# Requires: the reference visualization cell (after Step 4) to have been run.
# =============================================================================
import glob, torch, os
from IPython.display import display
from gen_model.train_base             import default_se3_conf, default_model_conf
from gen_model.train_unconditional    import SE3Module
from gen_model.diffusion.se3_diffuser import SE3Diffuser
from gen_model.inference_unconditional import run_reverse_sde
from gen_model.inference_conditional  import compute_coord_scale
from gen_model.data.residue_constants import restype_order
from gen_model.utils.structure_utils  import (
    rigids_to_ca_trajectory, save_ca_trajectory_pdb, as_mdtraj_trajectory,
)

# seqres is set by the reference visualization cell; load it here if skipped
if 'seqres' not in vars():
    import pandas as pd
    seqres = pd.read_csv('gen_model/splits/atlas.csv', index_col='name').loc[prot_cfg.name, 'seqres']

_device   = 'cuda' if torch.cuda.is_available() else 'cpu'
_ckpt_dir = f'checkpoints/{prot_cfg.name}_se3'
_ckpts    = [p for p in sorted(glob.glob(f'{_ckpt_dir}/*.ckpt')) if 'last' not in p]
_ckpt     = _ckpts[-1] if _ckpts else None

N_INLINE_SAMPLES = 5    # independent conformations to generate
N_INLINE_STEPS   = 100  # reverse-SDE steps (more = better quality, slower)

if _ckpt is None:
    print("No checkpoint found — train the model first (Step 8).")
    gen_ca = None
else:
    print(f"Checkpoint   : {_ckpt}")

    _se3_conf   = default_se3_conf()
    _model_conf = default_model_conf(
        lora_r=HPO_CONFIG['lora_r'], lora_alpha=HPO_CONFIG['lora_alpha']
    )
    _module = SE3Module.load_from_checkpoint(
        _ckpt, model_conf=_model_conf, se3_conf=_se3_conf, map_location=_device,
    ).to(_device).eval()

    _diffuser    = SE3Diffuser(_se3_conf)
    _coord_scale = compute_coord_scale(traj_path)
    _aatype      = torch.tensor([restype_order[c] for c in seqres], dtype=torch.long)
    _res_mask    = torch.ones(len(seqres), dtype=torch.float32)

    print(f"coord_scale  : {_coord_scale:.4f}")
    print(f"Generating {N_INLINE_SAMPLES} samples ({N_INLINE_STEPS} steps each) on {_device}…\n")

    _rigids = []
    for _s in range(N_INLINE_SAMPLES):
        _rigid = run_reverse_sde(
            model=_module, diffuser=_diffuser,
            aatype=_aatype, res_mask=_res_mask,
            N=len(seqres), num_steps=N_INLINE_STEPS, device=_device,
        )
        _rigids.append(_rigid)
        print(f"  Sample {_s + 1}/{N_INLINE_SAMPLES} done")

    gen_ca  = rigids_to_ca_trajectory(_rigids, _coord_scale)  # [N_SAMPLES, N, 3]
    gen_pdb = f'outputs/{prot_cfg.name}_generated.pdb'
    os.makedirs('outputs', exist_ok=True)
    save_ca_trajectory_pdb(gen_ca, gen_pdb, seqres)
    print(f"\nGenerated    : {gen_ca.shape}  (samples × residues × xyz, Å)")
    print(f"Saved PDB    : {gen_pdb}")

    # ── Inline 3-D comparison ─────────────────────────────────────────────────
    try:
        import nglview as nv

        # Reference (blue) — only if the reference cell was run
        if 'ref_ca' in vars() and ref_ca is not None:
            print("\n▶ Reference MD frames (blue) — use the player bar to animate:")
            _v_ref = nv.show_mdtraj(as_mdtraj_trajectory(ref_ca, seqres))
            _v_ref.add_representation('cartoon', color='blue')
            _v_ref.center()
            display(_v_ref)

        # Generated (orange)
        print("\n▶ Generated conformations (orange) — model samples from noise:")
        _v_gen = nv.show_mdtraj(as_mdtraj_trajectory(gen_ca, seqres))
        _v_gen.add_representation('cartoon', color='orange')
        _v_gen.center()
        display(_v_gen)

    except ImportError:
        print("\nnglview not installed — install with:  pip install nglview")
        print(f"Or open {gen_pdb} in PyMOL / VMD / UCSF ChimeraX")


In [ ]:
# Inference runs the reverse SE(3) SDE from pure noise to generate new conformations.
# Run from terminal after training completes:

npy_path = f'data/{prot_cfg.name}/{prot_cfg.name}_R{prot_cfg.replica}_latent.npy'

if best_ckpt:
    print("To generate samples, run from terminal:")
    print()
    print(f"  python gen_model/inference_unconditional.py \\")
    print(f"      --checkpoint {best_ckpt} \\")
    print(f"      --npy_path {npy_path} \\")
    print(f"      --protein_name {prot_cfg.name} \\")
    print(f"      --num_steps {protein_config.inference.num_steps} \\")
    print(f"      --num_samples {protein_config.inference.num_samples} \\")
    print(f"      --out_dir outputs/{prot_cfg.name}_se3")
else:
    print("Train a model first (Step 8), then run inference.")


## Step 10: Generate Samples

In [ ]:
import matplotlib.pyplot as plt, pandas as pd, glob

# Lightning's CSVLogger writes metrics to lightning_logs/version_N/metrics.csv
log_files = sorted(glob.glob('lightning_logs/version_*/metrics.csv'))

if log_files:
    df  = pd.read_csv(log_files[-1])
    val = df[['epoch', 'val_loss']].dropna()
    trn = df[['step',  'train_loss_step']].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    if not val.empty:
        axes[0].plot(val['epoch'], val['val_loss'], 'o-', color='steelblue')
        axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val Loss')
        axes[0].set_title(f'Validation Loss — {prot_cfg.name}'); axes[0].grid(alpha=0.3)
    if not trn.empty:
        axes[1].plot(trn['step'], trn['train_loss_step'], alpha=0.6, color='steelblue')
        axes[1].set_xlabel('Step'); axes[1].set_ylabel('Train Loss (step)')
        axes[1].set_title('Training Loss'); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    print(f"Log: {log_files[-1]}")
else:
    print("No training logs found. Run Step 8 (training) first.")
    print("Expected location: lightning_logs/version_*/metrics.csv")

## Step 11: Visualize

In [ ]:
import os, subprocess

save_zip = f'{prot_cfg.name}_results.zip'
dirs_to_pack = [d for d in [ckpt_dir, f'outputs/{prot_cfg.name}_se3'] if os.path.isdir(d)]

if dirs_to_pack:
    subprocess.run(['zip', '-rq', save_zip] + dirs_to_pack, check=True)
    print(f"✓ Packaged: {save_zip}  ({', '.join(dirs_to_pack)})")
    if IN_COLAB:
        from google.colab import files
        files.download(save_zip)
        print("✓ Download started")
    else:
        print(f"✓ Saved locally as {save_zip}")
else:
    print("Nothing to package yet — run Steps 8 and 10 first.")

## Step 12: Download Results

In [ ]:
# Package results
!zip -rq {prot_cfg.name}_results.zip {save_dir} {output_dir}

print(f"\nResults packaged: {prot_cfg.name}_results.zip")
print(f"  Checkpoints: {save_dir}/")
print(f"  Outputs: {output_dir}/")

if IN_COLAB:
    from google.colab import files
    files.download(f'{prot_cfg.name}_results.zip')
    print("\n✓ Download started")
else:
    print(f"\n✓ Saved locally as {prot_cfg.name}_results.zip")

## Summary

**What we did:**
1. ✓ Configured protein: `{protein_config.protein.name}_R{protein_config.protein.replica}`
2. ✓ Downloaded and preprocessed trajectory data using 
download_and_prep.py
3. ✓ Trained DDPM for {protein_config.training.num_epochs} epochs
4. ✓ Evaluated denoising on validation frames
5. ✓ Generated new conformations from noise

**Key insight:** The model learned the conformational space of **one specific protein** by training on different frames from its MD trajectory.

**To train on a different protein:**
- Change `protein_config.protein.name`
- Rerun from Step 3 onwards